# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [2]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [3]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = {
    row['id']: row 
    for row in (json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip())}
golden = {
    row['id']: row 
    for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets)

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'j01': {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}, 'j02': {'id': 'j02', 'snippet': "We're looking for a Data Analyst at Northwind Ltd. Reporting to the Head of Analytics, you'll work with SQL, dashboards, and stakeholder requests. 2 years minimum experience preferred."}, 'j03': {'id': 'j03', 'snippet': "Globex International seeks a Product Manager - Growth. You'll own activation and onboarding metrics across our consumer apps. We're looking for someone with at least 4 years of product management experience in B2C contexts."}, 'j04': {'id': 'j04', 'snippet': 'Join Initech as a Lead DevOps Engineer. We need a builder, not a maintainer. Around 6 years of hands-on infrastructure work expected, ideally with AWS, Terraform, and modern CI/CD.'}, 'j05': {'id': 'j

In [4]:
golden


{'j01': {'id': 'j01',
  'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5,
  'notes': 'clean — clear company + role + years'},
 'j02': {'id': 'j02',
  'company': 'Northwind Ltd',
  'role': 'Data Analyst',
  'years_experience_required': 2,
  'notes': 'clean — preferred used but still a stated minimum'},
 'j03': {'id': 'j03',
  'company': 'Globex International',
  'role': 'Product Manager - Growth',
  'years_experience_required': 4,
  'notes': "clean — 'at least 4 years'"},
 'j04': {'id': 'j04',
  'company': 'Initech',
  'role': 'Lead DevOps Engineer',
  'years_experience_required': 6,
  'notes': "slightly fuzzy — 'around 6 years' — accept 6"},
 'j05': {'id': 'j05',
  'company': 'Hooli',
  'role': 'Junior Frontend Developer',
  'years_experience_required': 0,
  'notes': 'edge — fresh grads OK = 0 years'},
 'j06': {'id': 'j06',
  'company': 'Pied Piper Inc.',
  'role': 'Senior ML Engineer',
  'years_experience_required': 7,
  'notes': "fuzzy — 

In [5]:
snippets

{'j01': {'id': 'j01',
  'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'},
 'j02': {'id': 'j02',
  'snippet': "We're looking for a Data Analyst at Northwind Ltd. Reporting to the Head of Analytics, you'll work with SQL, dashboards, and stakeholder requests. 2 years minimum experience preferred."},
 'j03': {'id': 'j03',
  'snippet': "Globex International seeks a Product Manager - Growth. You'll own activation and onboarding metrics across our consumer apps. We're looking for someone with at least 4 years of product management experience in B2C contexts."},
 'j04': {'id': 'j04',
  'snippet': 'Join Initech as a Lead DevOps Engineer. We need a builder, not a maintainer. Around 6 years of hands-on infrastructure work expected, ideally with AWS, Terraform, and modern CI/CD.'},
 'j05': {'id': 'j05',
  'snippet': 'Hooli is looking for a J

## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [6]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    # TODO: return a messages list like [{'role': 'user', 'content': '...'}]
    try: 
        content = (
        "Extract job details from the following job snippet with these keys:\n"
        '1.company, 2.role, 3.years_experience_required as integer, 4.notes'
        "Output valid JSON with no markdown code.\n\n"
        f'Input Snippet: "{snippet_text}"' )
        return [{'role': 'user', 'content': content}]
    except E:
        print(f"Exception: {e}")
    


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    # TODO: same shape, but include examples
    try: 
        content = (
        "Extract job details from the following job snippet with these keys:\n"
        '1.company, 2.role, 3.years_experience_required as integer, 4.notes'
        'Example 1:\n'
        'Input: "Globex International seeks a Senior Engineer with 5+ years of experience, joining timeframe immediate"\n'
        'Output: {"company": "Globex International", "role": "Senior Engineer", "years_experience_required": 5, "notes": "Software Engineer, Immediate joining."}\n\n'
        'Example 2:\n'
        'Input: "We want a tester, ~3 years in software testing, for our stealth startup."\n'
        'Output: {"company": "Unknown", "role": "Software Tester", "years_experience_required": 3, "notes": "Software Tester for sleath startup."}\n\n'
        'Example 3\n'
        'Input: "Looking for person for Sales Manager role on fastest growing company. The role will be inone of the Reliance Group of compaies."\n'
        'Output: {"company": "Reliance Group", "role": "Sales Manager", "years_experience_required": null, "notes": "Looking for Sales maanger position, since manager position might require 10+ years experience."}\n\n'
        f'Input snippet:\n"{snippet_text}"\n\n'
        "Output valid JSON with no markdown code.\n\n")
        return [{'role': 'user', 'content': content}]
    except E:
        print(f"Exception: {e}")
    


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    # TODO: 'You are an expert recruiter... Output JSON with these exact fields...'
    try:
        content = (
        "You are an expert data extraction assistant. Extract job requirements "
        "from the text into a JSON object strictly matching this structural specification:\n\n"
        "EXPECTED JSON SCHEMA:\n"
        "{\n"
        '  "company": string,                   // Canonical company name, or "Unknown"\n'
        '  "role": string,                      // Specific job title\n'
        '  "years_experience_required": int|null, // Integer >= 0, or null if unspecified/ambiguous\n'
        '  "notes": string                      // Explicit justification for nulls or extracted ranges\n'
        "}\n\n"
        "FIELD RULES:\n"
        "- For years of experience extract one integer lowest from the range\n"
        "- If experience is not specified set years_experience_required to null.\n"
        "- Do not guess company names if not explicitly mentioned.\n\n"
        "- Company name, set to 'Unknown' if missing"
        f'Input Snippet:\n"{snippet_text}"\n\n'
        "Output valid JSON with no markdown code.\n\n"
        )
        return [{"role": "user", "content": content}]
    except E:
        print(f"Exception: {e}")



def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    # TODO: 'Think step by step, then answer with JSON.'
    try:
        content = (
                "Extract job details from the snippet with keys: "
                '"company", "role", "years_experience_required", and "notes".\n\n'
                "Follow these steps before producing the output:\n"
                "1. Identify the company name (set to 'Unknown' if missing).\n"
                "2. Identify the role/job title.\n"
                "3. Extract required years of experience as an integer (or null if missing).\n"
                "4. Why this is done into 'notes' field.\n\n"
                f'Input Snippet: "{snippet_text}"\n\n'
                "Output valid JSON with no markdown code.\n\n:"
            )
        return [{"role": "user", "content": content}]
    except E:
        print(f"Exception: {e}")


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

In [7]:
resp = prompt_few_shot('Join Initech as a Lead DevOps Engineer. We need a builder, not a maintainer. Around 6 years of hands-on infrastructure work expected, ideally with AWS, Terraform, and modern CI/CD.')
print(f"Prompt_few_shot: {resp}")
resp = prompt_zero_shot('Join Initech as a Lead DevOps Engineer. We need a builder, not a maintainer. Around 6 years of hands-on infrastructure work expected, ideally with AWS, Terraform, and modern CI/CD.')
print(f"Prompt_Zero_shot: {resp}")
resp = prompt_structured('Join Initech as a Lead DevOps Engineer. We need a builder, not a maintainer. Around 6 years of hands-on infrastructure work expected, ideally with AWS, Terraform, and modern CI/CD.')
print(f"prompt_structured: {resp}")
resp = prompt_cot('Join Initech as a Lead DevOps Engineer. We need a builder, not a maintainer. Around 6 years of hands-on infrastructure work expected, ideally with AWS, Terraform, and modern CI/CD.')
print(f"prompt_cot: {resp}")


Prompt_few_shot: [{'role': 'user', 'content': 'Extract job details from the following job snippet with these keys:\n1.company, 2.role, 3.years_experience_required as integer, 4.notesExample 1:\nInput: "Globex International seeks a Senior Engineer with 5+ years of experience, joining timeframe immediate"\nOutput: {"company": "Globex International", "role": "Senior Engineer", "years_experience_required": 5, "notes": "Software Engineer, Immediate joining."}\n\nExample 2:\nInput: "We want a tester, ~3 years in software testing, for our stealth startup."\nOutput: {"company": "Unknown", "role": "Software Tester", "years_experience_required": 3, "notes": "Software Tester for sleath startup."}\n\nExample 3\nInput: "Looking for person for Sales Manager role on fastest growing company. The role will be inone of the Reliance Group of compaies."\nOutput: {"company": "Reliance Group", "role": "Sales Manager", "years_experience_required": null, "notes": "Looking for Sales maanger position, since man

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [ ]:
import sys
import importlib
from pathlib import Path
import json
from types import SimpleNamespace
from pydantic import BaseModel
import time

class Answer(BaseModel):
    question: str
    text:     str
    cost_usd: float
    retries:  int = 0
    confidence: float = 1.0
    sources: list[str] = []
    prompt_tokens: int
    completion_tokens: int

# Add project root directory to sys.path
#ROOT_DIR = Path(__file__).resolve().parents[3]

ROOT_DIR = Path.cwd().parents[3]
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Dynamically import the pipeline module using the exact directory name string
pipeline_module = importlib.import_module("IITM-AI-RAG.src.pipeline.pipeline")

_pipeline_ask_llm = pipeline_module.ask_llm
_pipeline_stream = pipeline_module.stream_answer

def parse_response(text: str | dict) -> int | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    # TODO: extract + parse the JSON, return a dict or None
    # Doing this as I've called pipeline.ask_llm and the result is returned as dict
    try:
        if isinstance(text, dict):
            return 1
        else:
            return 0
           # always parses now
        
    except:
        return  0


async def run_one(strategy_name: str, snippet: dict, snipID: str) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    # TODO: call the model, time the call, compute cost, parse the response
    question = STRATEGIES[strategy_name](snippet)
    content = SimpleNamespace(text=str(question[0]['content']))
    start_time = time.perf_counter()
    try:    
        response = await _pipeline_ask_llm(content) #Calling the function built within src.pipeline.pipeline = have attached the code in .MD
        response_dump = response.model_dump()
    except:
        response_dump = ""
    parse_status = parse_response(response_dump)
    if parse_status == 0:
        response_dump ={"text": "{}"}
    latency_s = time.perf_counter() - start_time
    #print(f"Response: {response} \n\n")
    response_dump["snippet_id"] = snipID
    response_dump["strategy"] = strategy_name
    response_dump["latency"] = latency_s
    response_dump['parse_status'] = parse_status
    print(f"Response: {response_dump} \n\n")
    return response_dump
    

async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    # TODO: build the task list, gather, return results
    tasks = []
    for item in snippets.values():
        print('Snippet {item}')
        for strategy in STRATEGIES.keys():
            tasks.append(run_one(strategy, item["snippet"], item["id"]))
            ##break
        ##break
    results = await asyncio.gather(*tasks)
    return results
    #raise NotImplementedError



In [9]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
print(f"Results: {results}")
results[0]

Snippet {item}
Snippet {item}
Snippet {item}
Snippet {item}
Snippet {item}
Snippet {item}
Snippet {item}
Snippet {item}
Snippet {item}
Snippet {item}


Raw structure: Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ZcnieZ1C9lqO0KjLK7WEOoVT', function=Function(arguments='{"content":"{\\"company\\": \\"Northwind Ltd.\\", \\"role\\": \\"Data Analyst\\", \\"years_experience_required\\": 2, \\"notes\\": \\"2 years minimum experience preferred.\\"}","confidence":0.95,"sources":[],"prompt_tokens":39,"completion_tokens":216}', name='answer_question'), type='function')]))
structured args: {'content': '{"company": "Northwind Ltd.", "role": "Data Analyst", "years_experience_required": 2, "notes": "2 years minimum experience preferred."}', 'confidence': 0.95, 'sources': [], 'prompt_tokens': 39, 'completion_tokens': 216}
confidence (from the MODEL): 0.95
Response: {'question': 'Extract job details from the snippet with keys: "company", "role", "years_experience_r

{'question': 'Extract job details from the following job snippet with these keys:\n1.company, 2.role, 3.years_experience_required as integer, 4.notesOutput valid JSON with no markdown code.\n\nInput Snippet: "Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems."',
 'text': '{"company": "Acme Corp", "role": "Senior Software Engineer", "years_experience_required": 5, "notes": "The ideal candidate has strong skills in Python and distributed systems."}',
 'cost_usd': 6.33e-05,
 'retries': 0,
 'confidence': 0.95,
 'sources': [],
 'prompt_tokens': 162,
 'completion_tokens': 65,
 'snippet_id': 'j01',
 'strategy': 'zero_shot',
 'latency': 2.376591009000549,
 'parse_status': 1}

In [10]:
for res in results:
    print(f"Data as {res['snippet_id']} | {res['strategy']} | {json.loads(res['text'])['company']} \n")

for gold in golden.values():
    print(f"Golden set as : {gold} \n")

for snipp in snippets.values():
    print(f"Snippets as : {snipp} \n")

Data as j01 | zero_shot | Acme Corp 

Data as j01 | few_shot | Acme Corp 

Data as j01 | structured | Acme Corp 

Data as j01 | cot | Acme Corp 

Data as j02 | zero_shot | Northwind Ltd. 

Data as j02 | few_shot | Northwind Ltd 

Data as j02 | structured | Northwind Ltd. 

Data as j02 | cot | Northwind Ltd. 

Data as j03 | zero_shot | Globex International 

Data as j03 | few_shot | Globex International 

Data as j03 | structured | Globex International 

Data as j03 | cot | Globex International 

Data as j04 | zero_shot | Initech 

Data as j04 | few_shot | Initech 

Data as j04 | structured | Initech 

Data as j04 | cot | Initech 

Data as j05 | zero_shot | Hooli 

Data as j05 | few_shot | Hooli 

Data as j05 | structured | Hooli 

Data as j05 | cot | Hooli 

Data as j06 | zero_shot | Pied Piper Inc. 

Data as j06 | few_shot | Pied Piper Inc. 

Data as j06 | structured | Pied Piper Inc. 

Data as j06 | cot | Pied Piper Inc. 

Data as j07 | zero_shot | Soylent Industries 

Data as j07 | 

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [11]:
RUBRIC = """You are a strict, fair evaluator of responses from a job snippet extraction assistant.
Score the CANDIDATE answer against the IDEAL response set on three dimensions, each 1-4:

- accuracy    : are the facts correct and complete versus the ideal? (1 Poor .. 4 Excellent)
- groundedness: is it supported by the ideal/source, with nothing invented or contradictory?
- format      : is it clear, appropriately concise, and well-structured?

Three fields: Company name, years_experience_required, role. Notes to have valid reasoning.

Scale: 1 = Poor (none correct or unparsable), 2 = OK (one of three correct, or fabricated a field), 
3 = Good (two of three correct, no fabricated data), 4 = Excellent (all three fields correct).
Be strict on accuracy: an answer that omits a key fact or contradicts the ideal cannot score above 2.
Return your scores and a one-paragraph reasoning that names specific facts."""

JUDGE_TOOL = {
    "type": "function",
    "function": {
        "name": "submit_scores",
        "description": "Submit rubric scores and reasoning for the candidate answer.",
        "parameters": {
            "type": "object",
            "properties": {
                "accuracy":     {"type": "integer", "minimum": 1, "maximum": 4},
                "groundedness": {"type": "integer", "minimum": 1, "maximum": 4},
                "format":       {"type": "integer", "minimum": 1, "maximum": 4},
                "reasoning":    {"type": "string"},
            },
            "required": ["accuracy", "groundedness", "format", "reasoning"],
        },
    },
}


def build_messages(snippet, extracted, gold):
    user = (f"JOB SNIPPET:\n{snippet}\n\n"
            f"EXTRACTED RESULT:\n{extracted}\n\n"
            f"IDEAL RESULT:\n{gold}")
    return [{"role": "system", "content": RUBRIC},
            {"role": "user",   "content": user}]

#print(build_messages(gold???, golden[0]["ideal_answer"], candidates["good"])[1]["content"])

In [12]:

def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    # TODO: count exact matches (with normalisation)
    raise NotImplementedError


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    # TODO: prompt the judge with both the gold and the extracted, ask for a 1-4 score
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    resp = client.chat.completions.create(
        model=JUDGE_MODEL, #"gpt-4o-mini",
        temperature=0,                       # judging should be as consistent as possible
        messages=build_messages(snippet_text, extracted, gold),
        tools=[JUDGE_TOOL],
        tool_choice={"type": "function", "function": {"name": "submit_scores"}},
      )
    return json.loads(resp.choices[0].message.tool_calls[0].function.arguments)

In [13]:
for res in results:
    print(f"Data as {res['snippet_id']} | {res['strategy']} | {json.loads(res['text'])} \n")

Data as j01 | zero_shot | {'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5, 'notes': 'The ideal candidate has strong skills in Python and distributed systems.'} 

Data as j01 | few_shot | {'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5, 'notes': 'Hiring for a Senior Software Engineer position in the platform team with backend development experience in Python and distributed systems.'} 

Data as j01 | structured | {'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5, 'notes': '5+ years specified, lowest integer extracted.'} 

Data as j01 | cot | {'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5, 'notes': 'The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'} 

Data as j02 | zero_shot | {'company': 'Northwind Ltd.', 'role': 'Data Analyst', 'years_experienc

In [14]:
# Apply scoring to all 40 results
# TODO: loop through results, attach accuracy + parse_success + llm_judge_score to each row
scored = []   # list of result dicts with scoring fields added

for result in results:
    txt = json.loads(result['text'])
    print(f"{snippets[result['snippet_id']]['snippet']}\n, {txt}, {result['strategy']}, \n {golden[result['snippet_id']]}\n\n")
    judge_result = await score_llm_judge(snippets[result['snippet_id']]['snippet'], txt, golden[result['snippet_id']] )
    judge_result['strategy'] = result['strategy']
    judge_result['snippet_id'] = result['snippet_id']
    judge_result['cost_usd'] = result['cost_usd']
    judge_result['latency'] = result['latency']
    judge_result['parse_status'] = result['parse_status']
    if judge_result['parse_status'] ==1:
        judge_result['llm_judge_score'] = judge_result['accuracy']*0.5+judge_result['groundedness']*0.3+judge_result['format']*0.15
    else:
        judge_result['llm_judge_score'] = 0

    print(f"Judge: {judge_result}")
    #print(f"Data as {res['snippet_id']} | {res['strategy']} | {json.loads(res['text'])['company']} \n")
    scored.append(judge_result)


print(f'Scored {len(scored)} results.')

Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.
, {'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5, 'notes': 'The ideal candidate has strong skills in Python and distributed systems.'}, zero_shot, 
 {'id': 'j01', 'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5, 'notes': 'clean — clear company + role + years'}




Judge: {'accuracy': 4, 'groundedness': 4, 'format': 4, 'reasoning': "The candidate's extracted result is fully accurate and matches the ideal result in all key aspects. The company name 'Acme Corp', the role 'Senior Software Engineer', and the years of experience required '5' are all correctly identified and match the ideal. The notes field is also consistent with the ideal, providing additional context about the skills required. The format is clear, concise, and well-structured, making it easy to read and understand. There are no inaccuracies or unsupported claims, thus earning a perfect score in all dimensions.", 'strategy': 'zero_shot', 'snippet_id': 'j01', 'cost_usd': 6.33e-05, 'latency': 2.376591009000549, 'parse_status': 1, 'llm_judge_score': 3.8000000000000003}
Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.
, {'company': 'Acme Corp', 

In [15]:
scored

[{'accuracy': 4,
  'groundedness': 4,
  'format': 4,
  'reasoning': "The candidate's extracted result is fully accurate and matches the ideal result in all key aspects. The company name 'Acme Corp', the role 'Senior Software Engineer', and the years of experience required '5' are all correctly identified and match the ideal. The notes field is also consistent with the ideal, providing additional context about the skills required. The format is clear, concise, and well-structured, making it easy to read and understand. There are no inaccuracies or unsupported claims, thus earning a perfect score in all dimensions.",
  'strategy': 'zero_shot',
  'snippet_id': 'j01',
  'cost_usd': 6.33e-05,
  'latency': 2.376591009000549,
  'parse_status': 1,
  'llm_judge_score': 3.8000000000000003},
 {'accuracy': 4,
  'groundedness': 4,
  'format': 4,
  'reasoning': "The candidate's extracted result is fully accurate and matches the ideal response. The company name 'Acme Corp', the role 'Senior Software 

## Step 5 — Build the comparison table

In [16]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_status': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency': 'median',
}).round(3)

summary.columns = ['Accuracy based on judge of all 3 fields', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

,Accuracy based on judge of all 3 fields,Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,3.5,1.0,3.370,0.001,1.591
few_shot,3.8,1.0,3.625,0.001,1.500
structured,4.0,1.0,3.800,0.001,1.435
zero_shot,3.5,1.0,3.385,0.001,1.740


In [17]:
df

,accuracy,groundedness,format,reasoning,strategy,snippet_id,cost_usd,latency,parse_status,llm_judge_score
0,4,4,4,The candidate's extracted result is fully accu...,zero_shot,j01,0.000063,2.376591,1,3.80
1,4,4,4,The candidate's extracted result is fully accu...,few_shot,j01,0.000099,1.637517,1,3.80
2,4,4,4,The candidate's extracted result matches the i...,structured,j01,0.000078,1.543426,1,3.80
3,4,4,4,The candidate's extracted result matches the i...,cot,j01,0.000077,1.551843,1,3.80
4,4,4,4,The candidate's extracted result matches the i...,zero_shot,j02,0.000071,1.632005,1,3.80
5,4,4,4,The candidate's extracted result matches the i...,few_shot,j02,0.000094,1.457990,1,3.80
6,4,4,4,The candidate's extracted result matches the i...,structured,j02,0.000077,1.979464,1,3.80
7,4,4,4,The candidate's extracted result matches the i...,cot,j02,0.000068,1.385971,1,3.80
8,4,4,4,The candidate's extracted result matches the i...,zero_shot,j03,0.000072,2.427495,1,3.80
9,4,4,4,The candidate's extracted result matches the i...,few_shot,j03,0.000096,1.624274,1,3.80


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```